In [ ]:
# set up environment
import sys
import json
import re
import requests
import pandas as pd
from pathlib import Path
import json
import csv
from caveclient import CAVEclient

import urllib.parse

from pathlib import Path
#pip install neuprint-python
from neuprint import Client, fetch_neurons, fetch_custom, NeuronCriteria as NC


print("Python executable:", sys.executable)
print("Imports OK")

c:\Users\jsfdz\Documents\Python\cave_env\lib\site-packages\neuprint\utils.py:89: UserWarning: 
Progress bar will not work well in the notebook without ipywidgets.
Run the following commands (for notebook and jupyterlab users):

    conda install -c conda-forge ipywidgets
    jupyter nbextension enable --py widgetsnbextension
    jupyter labextension install @jupyter-widgets/jupyterlab-manager

...and then reload your jupyter session, and restart your kernel.

  warnings.warn(msg)


Python executable: c:\Users\jsfdz\Documents\Python\cave_env\Scripts\python.exe
Imports OK


In [ ]:
# set up directories
PROJECT_ROOT = Path.cwd() #anchor to current notebook location


OUTPUT_DIR = PROJECT_ROOT / "output-MANCv1.2.3_04B_morphology_clusters_T1_20260407"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


print("Output:", OUTPUT_DIR)



Output: c:\Users\jsfdz\4B-analysis\4B-analysis\output-MANCv1.2.3_04B_morphology_clusters_T1_20260407


In [3]:
from dotenv import load_dotenv
import os

load_dotenv()  # loads .env from current working directory

token = os.environ["NEUPRINT_APPLICATION_CREDENTIALS"]

In [4]:
from neuprint import Client

c = Client(
    "https://neuprint.janelia.org",
    dataset="manc:v1.2.3",
    token=token
)
print(c)

Client("https://neuprint.janelia.org", "manc:v1.2.3")


In [5]:
#Fetch neurons to popualte the state later: one neuromere with 04B 

from neuprint import fetch_custom

cypher = """
MATCH (n:Neuron)
WHERE n.hemilineage = '04B'
  AND n.somaNeuromere = 'T1'
  AND n.somaSide = 'LHS'
RETURN
  n.bodyId AS bodyId,
  n.type AS type,
  n.instance AS instance,
  n.class AS class,
  n.subclass AS subclass,
  n.hemilineage AS hemilineage,
  n.somaNeuromere AS somaNeuromere,
  n.somaSide AS somaSide,
  n.rootSide AS rootSide,
  n.longTract AS longTract,
  n.birthtime AS birthtime
ORDER BY bodyId
"""

df_4b_t1_lhs = fetch_custom(cypher, client=c)
#df_4b_t1_lhs.head() 
#df_4b_t1_lhs["bodyId"].nunique() # n=97

In [7]:
# Generate a new state that diplays the 4B morphology clusters, as layers for each subclass WITH  all neurons in one layer displayed with the same color

import json
import urllib.parse
from itertools import cycle


#generate new, clean state:

##define sources:
# 1) Paste a known-good MANC Clio URL here (we wont repopulate it; it is jsut for retrieving sources)
BASE_URL = "https://clio-ng.janelia.org/#!%7B%22title%22:%22manc-v1.2.3-neuprint-layers%22%2C%22dimensions%22:%7B%22x%22:%5B8e-9%2C%22m%22%5D%2C%22y%22:%5B8e-9%2C%22m%22%5D%2C%22z%22:%5B8e-9%2C%22m%22%5D%7D%2C%22position%22:%5B23056.5%2C29733.5%2C41138.5%5D%2C%22crossSectionOrientation%22:%5B0%2C0.7071067690849304%2C-0.7071067690849304%2C0%5D%2C%22crossSectionScale%22:1%2C%22projectionOrientation%22:%5B0%2C0.7071067690849304%2C-0.7071067690849304%2C0%5D%2C%22projectionScale%22:91364.04452716278%2C%22layers%22:%5B%7B%22type%22:%22image%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-2-26-213dba213ef26e094c16c860ae7f4be0/v3_emdata_clahe_xy/jpeg%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22tab%22:%22rendering%22%2C%22name%22:%22em%22%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22toolBindings%22:%7B%22Q%22:%22selectSegments%22%7D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2210007%22%5D%2C%22segmentColors%22:%7B%2233946%22:%22#808080%22%7D%2C%22name%22:%22manc:v1.2.3%22%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-roi-d5f392696f7a48e27f49fa1a9db5ee3b/all-vnc-roi%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22pick%22:false%2C%22tab%22:%22rendering%22%2C%22selectedAlpha%22:0%2C%22saturation%22:0%2C%22meshSilhouetteRendering%22:4%2C%22segments%22:%5B%221%22%5D%2C%22segmentDefaultColor%22:%22#ffffff%22%2C%22name%22:%22all-tissue%22%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-roi-d5f392696f7a48e27f49fa1a9db5ee3b/synaptic-neuropil%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22pick%22:false%2C%22tab%22:%22rendering%22%2C%22selectedAlpha%22:0%2C%22saturation%22:0%2C%22meshSilhouetteRendering%22:4%2C%22segments%22:%5B%221%22%5D%2C%22segmentDefaultColor%22:%22#ffffff%22%2C%22segmentColors%22:%7B%221%22:%22#ffffff%22%7D%2C%22name%22:%22all-synaptic%22%2C%22archived%22:true%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-roi-d5f392696f7a48e27f49fa1a9db5ee3b/roi-202208%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22pick%22:false%2C%22tab%22:%22segments%22%2C%22selectedAlpha%22:0%2C%22saturation%22:0.5%2C%22meshSilhouetteRendering%22:4%2C%22segments%22:%5B%2210%22%2C%2211%22%2C%2212%22%2C%2213%22%2C%2214%22%2C%2215%22%2C%2216%22%2C%2217%22%2C%2218%22%2C%2219%22%2C%2220%22%2C%2221%22%2C%2222%22%2C%2223%22%2C%2224%22%2C%2225%22%2C%2226%22%2C%2227%22%2C%224%22%2C%225%22%2C%226%22%2C%227%22%2C%228%22%2C%229%22%5D%2C%22name%22:%22neuropils%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-roi-d5f392696f7a48e27f49fa1a9db5ee3b/nerve-roi-202301%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22pick%22:false%2C%22tab%22:%22segments%22%2C%22objectAlpha%22:0.1%2C%22segments%22:%5B%221%22%2C%2210%22%2C%2211%22%2C%2212%22%2C%2213%22%2C%2214%22%2C%2215%22%2C%2216%22%2C%2217%22%2C%2218%22%2C%2219%22%2C%222%22%2C%2220%22%2C%2221%22%2C%2222%22%2C%2223%22%2C%2224%22%2C%2225%22%2C%2226%22%2C%2227%22%2C%2228%22%2C%2229%22%2C%223%22%2C%2230%22%2C%2231%22%2C%2232%22%2C%2233%22%2C%2234%22%2C%2235%22%2C%224%22%2C%225%22%2C%226%22%2C%227%22%2C%228%22%2C%229%22%5D%2C%22name%22:%22nerves%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22annotation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-v1.2-synapse-partners-minconf-0.0.precomputed%22%2C%22tab%22:%22rendering%22%2C%22ignoreNullSegmentFilter%22:false%2C%22shader%22:%22#uicontrol%20bool%20showPre%20checkbox%28default=true%29%5Cn#uicontrol%20bool%20showPostPartners%20checkbox%28default=true%29%5Cn%5Cn#uicontrol%20vec3%20preColor%20color%28default=%5C%22red%5C%22%29%5Cn#uicontrol%20vec3%20postColor%20color%28default=%5C%22blue%5C%22%29%5Cn%5Cn//%20Note:%5Cn//%20%20%20For%20the%20MANC%20in%20neuprint%2C%5Cn//%20%20%20connection%20strengths%20don%27t%5Cn//%20%20%20include%20any%20synapses%20below%5Cn//%20%20%20confidence%200.4.%5Cn#uicontrol%20float%20preConfidence%20slider%28min=0%2C%20max=1%2C%20default=0%29%5Cn#uicontrol%20float%20postConfidence%20slider%28min=0%2C%20max=1%2C%20default=0%29%5Cn%5Cnvoid%20main%28%29%20%7B%5Cn%20%20setColor%28defaultColor%28%29%29%3B%5Cn%5Cn%20%20float%20preSize%20=%206.0%3B%5Cn%20%20float%20postSize%20=%204.0%3B%5Cn%20%20%5Cn%20%20setEndpointMarkerColor%28%5Cn%20%20%20%20vec4%28preColor%2C%200.5%29%2C%5Cn%20%20%20%20vec4%28postColor%2C%200.5%29%29%3B%5Cn%20%20%5Cn%20%20if%20%28showPre%20&&%20showPostPartners%29%20%7B%5Cn%20%20%20%20setLineWidth%281.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%28preSize%2C%20postSize%29%3B%5Cn%20%20%7D%5Cn%20%20else%20if%20%28showPostPartners%29%20%7B%5Cn%20%20%20%20setLineWidth%280.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%280.0%2C%20postSize%29%3B%5Cn%20%20%7D%5Cn%20%20else%20if%20%28showPre%29%20%7B%5Cn%20%20%20%20setLineWidth%280.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%28preSize%2C%200.0%29%3B%5Cn%20%20%7D%5Cn%20%20else%20%7B%5Cn%20%20%20%20setLineWidth%280.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%280.0%2C%200.0%29%3B%5Cn%20%20%7D%5Cn%20%20%5Cn%20%20if%20%28prop_pre_synaptic_confidence%28%29%20%3C%20preConfidence%20%7C%7C%5Cn%20%20%20%20%20%20prop_post_synaptic_confidence%28%29%20%3C%20postConfidence%29%20discard%3B%5Cn%7D%5Cn%22%2C%22shaderControls%22:%7B%22showPostPartners%22:false%2C%22postColor%22:%22#00ffff%22%2C%22preConfidence%22:0.4%2C%22postConfidence%22:0.4%7D%2C%22linkedSegmentationLayer%22:%7B%22pre_synaptic_cell%22:%22manc:v1.2.3%22%7D%2C%22filterBySegmentation%22:%5B%22pre_synaptic_cell%22%5D%2C%22name%22:%22presyn%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22annotation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-v1.2-synapse-partners-minconf-0.0.precomputed%22%2C%22tab%22:%22rendering%22%2C%22ignoreNullSegmentFilter%22:false%2C%22shader%22:%22#uicontrol%20bool%20showPrePartners%20checkbox%28default=true%29%5Cn#uicontrol%20bool%20showPost%20checkbox%28default=true%29%5Cn%5Cn#uicontrol%20vec3%20preColor%20color%28default=%5C%22red%5C%22%29%5Cn#uicontrol%20vec3%20postColor%20color%28default=%5C%22blue%5C%22%29%5Cn%5Cn//%20Note:%5Cn//%20%20%20For%20the%20MANC%20in%20neuprint%2C%5Cn//%20%20%20connection%20strengths%20don%27t%5Cn//%20%20%20include%20any%20synapses%20below%5Cn//%20%20%20confidence%200.4.%5Cn#uicontrol%20float%20preConfidence%20slider%28min=0%2C%20max=1%2C%20default=0.4%29%5Cn#uicontrol%20float%20postConfidence%20slider%28min=0%2C%20max=1%2C%20default=0.4%29%5Cn%5Cnvoid%20main%28%29%20%7B%5Cn%20%20setColor%28defaultColor%28%29%29%3B%5Cn%5Cn%20%20float%20preSize%20=%206.0%3B%5Cn%20%20float%20postSize%20=%204.0%3B%5Cn%20%20%5Cn%20%20setEndpointMarkerColor%28%5Cn%20%20%20%20vec4%28preColor%2C%200.5%29%2C%5Cn%20%20%20%20vec4%28postColor%2C%200.5%29%29%3B%5Cn%20%20%5Cn%20%20if%20%28showPrePartners%20&&%20showPost%29%20%7B%5Cn%20%20%20%20setLineWidth%281.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%28preSize%2C%20postSize%29%3B%5Cn%20%20%7D%5Cn%20%20else%20if%20%28showPost%29%20%7B%5Cn%20%20%20%20setLineWidth%280.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%280.0%2C%20postSize%29%3B%5Cn%20%20%7D%5Cn%20%20else%20if%20%28showPrePartners%29%20%7B%5Cn%20%20%20%20setLineWidth%280.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%28preSize%2C%200.0%29%3B%5Cn%20%20%7D%5Cn%20%20else%20%7B%5Cn%20%20%20%20setLineWidth%280.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%280.0%2C%200.0%29%3B%5Cn%20%20%7D%5Cn%20%20%5Cn%20%20if%20%28prop_pre_synaptic_confidence%28%29%20%3C%20preConfidence%20%7C%7C%5Cn%20%20%20%20%20%20prop_post_synaptic_confidence%28%29%20%3C%20postConfidence%29%20discard%3B%5Cn%7D%5Cn%22%2C%22shaderControls%22:%7B%22showPrePartners%22:false%2C%22postColor%22:%22#00ffff%22%7D%2C%22linkedSegmentationLayer%22:%7B%22post_synaptic_cell%22:%22manc:v1.2.3%22%7D%2C%22filterBySegmentation%22:%5B%22post_synaptic_cell%22%5D%2C%22name%22:%22postsyn%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://vnc-v3-seg-3d2f1c08fd4720848061f77362dc6c17/mask%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22pick%22:false%2C%22tab%22:%22segments%22%2C%22segmentColors%22:%7B%221%22:%22#000000%22%7D%2C%22name%22:%22voxel-classes%22%2C%22archived%22:true%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://vnc-v3-seg-3d2f1c08fd4720848061f77362dc6c17/rc5_wsexp%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22tab%22:%22segments%22%2C%22name%22:%22initial-supervoxels%22%2C%22archived%22:true%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-roi-d5f392696f7a48e27f49fa1a9db5ee3b/court-et-al-systematic-manc_tracts/mesh%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22tab%22:%22rendering%22%2C%22saturation%22:0.5%2C%22meshSilhouetteRendering%22:4%2C%22segments%22:%5B%22100%22%2C%22101%22%2C%22102%22%2C%22103%22%2C%22104%22%2C%22105%22%2C%22106%22%5D%2C%22name%22:%22court%20et%20al.%20tracts%22%2C%22archived%22:true%7D%5D%2C%22showSlices%22:false%2C%22prefetch%22:false%2C%22selectedLayer%22:%7B%22flex%22:1.49%2C%22size%22:426%2C%22visible%22:true%2C%22layer%22:%22manc:v1.2.3%22%7D%2C%22projectionBackgroundColor%22:%22#ffffff%22%2C%22layout%22:%223d%22%2C%22settingsPanel%22:%7B%22visible%22:true%7D%2C%22selection%22:%7B%22flex%22:0.51%2C%22size%22:426%2C%22visible%22:false%7D%7D"

# 2) Decode its state
encoded_state = BASE_URL.split("#!", 1)[1]
base_state = json.loads(urllib.parse.unquote(encoded_state))

# 3) Extract the actual working sources from that scene
em_source = None
seg_source = None

for L in base_state["layers"]:
    if L.get("type") == "image" and em_source is None:
        em_source = L.get("source")
    if L.get("type") == "segmentation" and seg_source is None:
        seg_source = L.get("source")

print("EM source:", em_source)
print("Seg source:", seg_source)

##generate state
new_state = {
    "title": "MANC_v1.2.3_4B_morphology_clusters_JHS",
    "dimensions": base_state["dimensions"],
    "position": base_state["position"],
    "crossSectionOrientation": base_state["crossSectionOrientation"],
    "crossSectionScale": base_state["crossSectionScale"],
    "projectionOrientation": base_state["projectionOrientation"],
    "projectionScale": base_state["projectionScale"],
    "layout": base_state.get("layout", "3d"),
    "showSlices": base_state.get("showSlices", True),
    "projectionBackgroundColor": base_state.get("projectionBackgroundColor", "#ffffff"),
    "crossSectionBackgroundColor": base_state.get("crossSectionBackgroundColor", "#ffffff"),
    "selectedLayer": base_state.get("selectedLayer"),
    "settingsPanel": base_state.get("settingsPanel", {"visible": True}),
    "layers": [
        {
            "type": "image",
            "name": "EM",
            "source": em_source,
            "visible": True
        }
    ]
}


# --- Color palette (cycles if needed) ---
COLORS = [
    "#87CEEB",  # sky blue
    "#FFA500",  # orange
    "#E34234",  # vermillion
    "#CC99FF",  # pale violet
    "#66CDAA",  # medium aquamarine
    "#FFD700",  # gold
]

color_cycle = cycle(COLORS)


# --- Helper: upsert layer (FIXED) ---
def add_segmentation_layer(state, name, body_ids, seg_source, color):
    body_ids = sorted(set(int(x) for x in body_ids))

    layer = {
        "type": "segmentation",
        "name": name,
        "source": seg_source,
        "segments": [str(x) for x in body_ids],
        "segmentColors": {str(x): color for x in body_ids},
        "visible": True,
    }

    # replace if exists
    for i, existing in enumerate(state["layers"]):
        if existing.get("name") == name:
            state["layers"][i] = layer
            return

    # otherwise append
    state["layers"].append(layer)


# --- Clean subclass labels ---
subclass_df = df_4b_t1_lhs.copy()
subclass_df["subclass"] = (
    subclass_df["subclass"]
    .fillna("unclassified")
    .astype(str)
    .str.strip()
)
subclass_df.loc[subclass_df["subclass"] == "", "subclass"] = "unclassified"


# --- Inspect counts ---
subclass_counts = (
    subclass_df.groupby("subclass")["bodyId"]
    .nunique()
    .sort_values(ascending=False)
)
print(subclass_counts)


# --- Build layers with colors ---
for subclass_name, group in subclass_df.groupby("subclass", sort=True):
    body_ids = group["bodyId"].dropna().astype(int).tolist()
    if not body_ids:
        continue

    layer_name = f"MANC_v1.2.3_{subclass_name}"
    color = next(color_cycle)

    add_segmentation_layer(new_state, layer_name, body_ids, seg_source, color)


#add more manual layers based on my cluster review. 
##use a dictionary and loop to add the layers:
###these came from notebook 4

br1_ids = [10024, 12523, 15309, 16487, 16527, 17351, 17409, 17534, 18571, 21956, 101447, 152244, 162540]
br2_ids = [13506, 14728, 14774, 15005, 18148, 19250, 19366, 13589, 13455]
br3_ids = [13535, 15002, 18755, 155236]

ir_2_04B_1_ids = [15811, 16757, 17572, 18785, 166421]
ir_2_04B_2_ids = [17678, 17935, 18605, 18831, 21499, 21808, 22163, 26941, 29988, 100339, 100525, 153878, 164146, 165318] 
ir_2_04B_3_ids = [20077, 101453] #cranial nuclei, ,primary neurite similar to 4 but braches gou along anterior neuropil 
ir_2_04B_4_ids =[20932, 21567, 36722, 38071, 158391,102138] #posterior nuclei,primary neurite similar to 3 but braches go along caudal neuropil 
ir_2_04B_5_ids = [18641, 20836] 
ir_2_04B_6_ids = [14389] 

ir_1_1_ids = [13770, 16224, 20585, 23790, 26388, 102536] #independent leg
ir_1_2_ids = [13128, 13409, 14697, 13798] 
ir_1_3_ids = [13975, 14024, 14954, 19833, 21189, 21322, 24181] 
ir_1_4_ids = [17567, 18156, 18253, 18629, 18990, 22868, 23716] 
ir_1_5_ids = [21862, 152655] 
ir_1_6_ids = [18724, 19985, 27760,22997] #add 22997

manual_layers = {
    "br1": br1_ids,
    "br2": br2_ids,
    "br3": br3_ids,
    "ir_2_04B_1": ir_2_04B_1_ids,
    "ir_2_04B_2": ir_2_04B_2_ids,
    "ir_2_04B_3": ir_2_04B_3_ids,
    "ir_2_04B_4": ir_2_04B_4_ids,
    "ir_2_04B_5": ir_2_04B_5_ids,
    "ir_2_04B_6": ir_2_04B_6_ids,
    "ir_1_1": ir_1_1_ids,
    "ir_1_2": ir_1_2_ids,
    "ir_1_3": ir_1_3_ids,
    "ir_1_4": ir_1_4_ids,
    "ir_1_5": ir_1_5_ids,
    "ir_1_6":ir_1_6_ids,

   
}

for name, ids in manual_layers.items():
    layer_name = f"MANC_v1.2.3_{name}"
    color = next(color_cycle)  # or set manually if you want control

    add_segmentation_layer(new_state, layer_name, ids, seg_source, color)



#Add asingle layer manually
# add_segmentation_layer(
#     new_state,
#     "MANC_v1.2.3_candidate_pair",
#     [123456789, 987654321],
#     seg_source,
#     "#FF0000"
# )



# --- Debug print ---
print("State title:", new_state["title"])
print("Number of layers:", len(new_state["layers"]))

for layer in new_state["layers"]:
    print(layer["name"], len(layer.get("segments", [])))


# --- Save ---

STATE_OUT = OUTPUT_DIR / "MANC_v1.2.3_4B_morphology_clusters_reviewed20260407.json"

with open(STATE_OUT, "w", encoding="utf-8") as f:
    json.dump(new_state, f, indent=2)


URL_OUT = OUTPUT_DIR / "MANC_v1.2.3_4B_morphology_clusters_reviewed20260407_url.txt"

with open(URL_OUT, "w") as f:
    f.write(NEW_URL)

print("Saved URL to:", URL_OUT.resolve())




# --- Generate Neuroglancer URL ---
encoded = urllib.parse.quote(json.dumps(new_state, separators=(",", ":")))
NEW_URL = "https://clio-ng.janelia.org/#!" + encoded

print("Saved new state:", STATE_OUT)
print("\nNew URL:\n")
print(NEW_URL)

EM source: {'url': 'precomputed://gs://flyem-vnc-2-26-213dba213ef26e094c16c860ae7f4be0/v3_emdata_clahe_xy/jpeg', 'subsources': {'default': True}, 'enableDefaultSubsources': False}
Seg source: [{'url': 'precomputed://gs://manc-seg-v1p2/manc-seg-v1.2', 'subsources': {'default': True, 'mesh': True}, 'enableDefaultSubsources': False}, {'url': 'precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/', 'subsources': {'default': True}, 'enableDefaultSubsources': False}, {'url': 'precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/', 'subsources': {'default': True}, 'enableDefaultSubsources': False}, {'url': 'precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/', 'enableDefaultSubsources': False}]
subclass
IR    60
BR    26
BI     6
IA     4
BA     1
Name: bodyId, dtype: int64
State title: MANC_v1.2.3_4B_morphology_clusters_JHS
Number of layers: 21
EM 0
MANC_v1.2.3_BA 1
MANC_v1.2.3_BI 6
MANC_v1.2.3_BR 26
MANC_v1.2